# NBA Game Prediction (Modular)

This notebook runs the modular NBA pipeline in this repo.

Rules for outputs:
- Predictions are logged to SQLite only (no JSON/CSV).
- Reports are displayed inline (no HTML files).

Prereqs:
- `pip install -r requirements.txt`
- Internet access for `nba_api` fetches

In [18]:
from __future__ import annotations

import sys
from datetime import datetime
from pathlib import Path

import pandas as pd

def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for p in [start] + list(start.parents):
        if (p / 'README.md').exists() and (p / 'data').exists():
            return p
    return start

ROOT = find_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

DB_PATH = str(ROOT / 'sports_analytics.db')
MODEL_DIR = ROOT / 'machine_learning' / 'models'

print('ROOT:', ROOT)
print('DB_PATH:', DB_PATH)
print('MODEL_DIR:', MODEL_DIR)

ROOT: C:\Users\Windows User\My_folder\Sports_Analytics
DB_PATH: C:\Users\Windows User\My_folder\Sports_Analytics\sports_analytics.db
MODEL_DIR: C:\Users\Windows User\My_folder\Sports_Analytics\machine_learning\models


In [19]:
from glob import glob

def _latest(glob_pattern: str) -> str | None:
    paths = sorted(glob(glob_pattern))
    return paths[-1] if paths else None

def _model_search_dirs(model_dir: Path = MODEL_DIR) -> list[Path]:
    # Support both canonical and legacy notebook-local model folders.
    candidates = [
        Path(model_dir),
        ROOT / 'basketball' / 'machine_learning' / 'models',
        Path.cwd().resolve() / 'machine_learning' / 'models',
    ]
    deduped: list[Path] = []
    seen: set[str] = set()
    for c in candidates:
        key = str(c.resolve()) if c.exists() else str(c)
        if key not in seen:
            seen.add(key)
            deduped.append(c)
    return deduped

def find_latest_model_paths(model_dir: Path = MODEL_DIR) -> dict:
    dirs = _model_search_dirs(model_dir)

    def _pick(pattern: str) -> str | None:
        for d in dirs:
            p = _latest(str(d / pattern))
            if p:
                return p
        return None

    gp = _pick('gp_*.pkl')
    lgbm_win = _pick('lgbm_win_*.pkl')
    lgbm_quantile = _pick('lgbm_quantile_*.pkl')
    elo = _pick('elo_*.pkl')
    return {
        'gp': gp,
        'lgbm_win': lgbm_win,
        'lgbm_quantile': lgbm_quantile,
        'elo': elo,
    }

print('Model search dirs:')
for d in _model_search_dirs():
    print(' -', d)

paths = find_latest_model_paths()
paths

Model search dirs:
 - C:\Users\Windows User\My_folder\Sports_Analytics\machine_learning\models
 - C:\Users\Windows User\My_folder\Sports_Analytics\basketball\machine_learning\models


{'gp': 'C:\\Users\\Windows User\\My_folder\\Sports_Analytics\\machine_learning\\models\\gp_v20260328_205655.pkl',
 'lgbm_win': 'C:\\Users\\Windows User\\My_folder\\Sports_Analytics\\machine_learning\\models\\lgbm_win_v20260328_205655.pkl',
 'lgbm_quantile': 'C:\\Users\\Windows User\\My_folder\\Sports_Analytics\\machine_learning\\models\\lgbm_quantile_v20260328_205655.pkl',
 'elo': 'C:\\Users\\Windows User\\My_folder\\Sports_Analytics\\machine_learning\\models\\elo_v20260328_205655.pkl'}

## Automatic full retrain




This notebook retrains automatically when:


- Model artifacts are missing, or


- The prediction-batch counter in SQLite (`retraining_metadata.incremental_count`) is a multiple of 7 (7, 14, 21, …).




The counter increments by 1 after each successful prediction batch (one run/day), not per game.

In [20]:
FAST_RETRAIN = True
RETRAIN_EVERY_N = int(globals().get('RETRAIN_EVERY_N', 7) or 7)
FORCE_RETRAIN = bool(globals().get('FORCE_RETRAIN', False))

paths = globals().get('paths') or find_latest_model_paths()
missing = [k for k, v in (paths or {}).items() if not v]

from data.database.database_handler import SportsAnalyticsDB
with SportsAnalyticsDB(DB_PATH) as db:
    state = db.get_retraining_state()
count = int(state.get('incremental_count') or 0)

scheduled = count > 0 and (count % RETRAIN_EVERY_N == 0)
should_retrain = FORCE_RETRAIN or bool(missing) or scheduled

if should_retrain:
    if FORCE_RETRAIN:
        reason = 'manual force retrain'
    elif missing:
        reason = 'missing artifacts: ' + ', '.join(missing)
    else:
        reason = f'scheduled (count % {RETRAIN_EVERY_N} == 0, count={count})'

    print('Full retrain triggered ->', reason)

    # Clear trainer module cache before importing (to pick up latest code changes)
    for key in list(sys.modules.keys()):
        if key.startswith('training.trainer') or 'training.trainer' in key:
            del sys.modules[key]

    from training.trainer import ModelTrainer

    trainer = ModelTrainer(db_path=DB_PATH)
    train_result = trainer.full_retrain(verbose=True, fast_mode=FAST_RETRAIN)
    print('Trained model_version:', train_result.get('model_version'))
    paths = train_result.get('model_paths', paths)

    # If the optional helper is defined in-session, print retrain accuracy snapshot.
    if '_print_retrain_accuracy_snapshot' in globals():
        _print_retrain_accuracy_snapshot(DB_PATH)
    else:
        print('Accuracy snapshot helper not defined in this session; run the feedback cell for metrics.')
else:
    print(f'No retrain needed. count={count}, missing={missing}, threshold={RETRAIN_EVERY_N}')

paths

No retrain needed. count=4, missing=[], threshold=7


{'gp': 'C:\\Users\\Windows User\\My_folder\\Sports_Analytics\\machine_learning\\models\\gp_v20260328_205655.pkl',
 'lgbm_win': 'C:\\Users\\Windows User\\My_folder\\Sports_Analytics\\machine_learning\\models\\lgbm_win_v20260328_205655.pkl',
 'lgbm_quantile': 'C:\\Users\\Windows User\\My_folder\\Sports_Analytics\\machine_learning\\models\\lgbm_quantile_v20260328_205655.pkl',
 'elo': 'C:\\Users\\Windows User\\My_folder\\Sports_Analytics\\machine_learning\\models\\elo_v20260328_205655.pkl'}

In [21]:
# Force fresh reload of database modules to pick up code changes
import sys
import importlib

# Remove cached modules (including new vectorized features module)
modules_to_reload = [
    'data.database.database_handler',
    'evaluators.prediction_logger',
    'src.evaluation.feedback_loop',
    'src.evaluation.vectorized_features',
    'training.trainer',
]

for mod_name in modules_to_reload:
    for key in list(sys.modules.keys()):
        if key.startswith(mod_name) or mod_name in key:
            del sys.modules[key]

print("✓ Module cache cleared - fresh imports will occur")
print("✓ Phase 2 vectorization module loaded")


✓ Module cache cleared - fresh imports will occur
✓ Phase 2 vectorization module loaded


## Generate predictions for upcoming games

This loads the latest saved models, builds leakage-safe rolling stats, predicts upcoming games, logs predictions to SQLite, and displays the HTML report inline.

In [22]:
from typing import Any

import numpy as np

from data.nba_loader import (
    fetch_nba_games,
    fetch_upcoming_games,
    get_all_nba_teams,
    get_team_latest_stats,
    prepare_prediction_features,
)
from data.feature_engineering import calculate_rolling_stats, prepare_training_data

from data.database.database_handler import SportsAnalyticsDB
from ensemble.ensemble_predictor import EnsemblePredictor
from src.evaluation.feedback_loop import (
    HIGH_SIGNAL_FEATURE_COLUMNS,
    PredictionFeedbackManager,
    derive_game_features,
)

def _to_date_str(value) -> str:
    try:
        return pd.to_datetime(value).date().isoformat()
    except Exception:
        return datetime.now().date().isoformat()

def _to_native(value: Any):
    if isinstance(value, dict):
        return {k: _to_native(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [_to_native(v) for v in value]
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, np.generic):
        return value.item()
    return value

def _models_missing(model_paths: dict) -> list[str]:
    return [k for k, v in (model_paths or {}).items() if not v]

def _build_numeric_features(game: dict, team_history_df: pd.DataFrame, fallback: dict | None = None) -> dict:
    derived = derive_game_features(game=game, team_history_df=team_history_df)
    out = {k: float(derived.get(k, 0.0)) for k in HIGH_SIGNAL_FEATURE_COLUMNS}
    if fallback:
        out.update({k: float(fallback.get(k, out.get(k, 0.0))) for k in HIGH_SIGNAL_FEATURE_COLUMNS})
    return out

# 1) Ensure models exist (self-heal by retraining if needed)
# Set FORCE_RETRAIN = True in a cell above to force a full retrain now.
FORCE_RETRAIN = bool(globals().get('FORCE_RETRAIN', False))
paths = globals().get('paths') or find_latest_model_paths()
missing = _models_missing(paths)
if FORCE_RETRAIN or missing:
    if FORCE_RETRAIN:
        print('FORCE_RETRAIN=True -> running full retrain before predictions...')
    else:
        print('Missing model artifacts:', missing)
    print('Running full retrain automatically...')
    from training.trainer import ModelTrainer

    trainer = ModelTrainer(db_path=DB_PATH)
    train_result = trainer.full_retrain(verbose=True)
    paths = train_result.get('model_paths', paths)

# 2) Load models
ep = EnsemblePredictor(model_dir=str(MODEL_DIR))
ep.load_models(
    gp_path=paths['gp'],
    lgbm_win_path=paths['lgbm_win'],
    lgbm_quantile_path=paths['lgbm_quantile'],
    elo_path=paths['elo'],
)
cal_status = ep.fit_calibrator_from_db(db_path=DB_PATH, limit=200, min_samples=10, save=True)
print('Calibrator fit status:', cal_status)

# 3) Historical game logs -> rolling stats -> matchup feature schema
games_df = fetch_nba_games(seasons=['2023-24', '2024-25'], season_type='Regular Season', verbose=True)
games_with_stats = calculate_rolling_stats(games_df, window=5)
matchup_df, _, baseline_feature_cols = prepare_training_data(games_with_stats, verbose=True)

model_feature_cols = list(getattr(getattr(ep, 'lgbm_win', None), 'feature_names', None) or baseline_feature_cols)
print('Using model feature columns:', len(model_feature_cols))

# Settled-only history used by derive_game_features
team_history_df = games_df.copy()
if 'PTS' in team_history_df.columns:
    team_history_df = team_history_df[pd.to_numeric(team_history_df['PTS'], errors='coerce').notna()].copy()
if 'GAME_DATE' in team_history_df.columns:
    team_history_df['GAME_DATE'] = pd.to_datetime(team_history_df['GAME_DATE'], errors='coerce')

# 4) Upcoming schedule (game_date is UTC from scoreboard endpoint)
upcoming = fetch_upcoming_games(verbose=True)
team_names = get_all_nba_teams().get('names', {})

predictions = []
logged_prediction_ids = []

with SportsAnalyticsDB(DB_PATH) as db:
    model_version = (db.get_retraining_state().get('model_version') or '')
    feedback = PredictionFeedbackManager(db, model_version=model_version)

    for game in upcoming:
        home_id = game.get('home_team_id')
        away_id = game.get('away_team_id')
        if not home_id or not away_id:
            continue

        home_id = int(home_id)
        away_id = int(away_id)

        home_stats = get_team_latest_stats(games_with_stats, home_id)
        away_stats = get_team_latest_stats(games_with_stats, away_id)
        if not home_stats or not away_stats:
            continue

        home_name = team_names.get(home_id, str(home_id))
        away_name = team_names.get(away_id, str(away_id))

        game_ctx = {
            'game_date': game.get('game_date'),
            'game_date_utc': game.get('game_date'),
            'home_team_id': home_id,
            'away_team_id': away_id,
            'home_team': home_name,
            'away_team': away_name,
        }
        derived_features = _build_numeric_features(game=game_ctx, team_history_df=team_history_df)

        features_df = prepare_prediction_features(home_stats, away_stats, model_feature_cols)
        for col in model_feature_cols:
            if col in derived_features:
                features_df[col] = float(derived_features[col])
        features_df = features_df.reindex(columns=model_feature_cols, fill_value=0.0)

        out = ep.predict(home_id, away_id, features_df)

        pred = {
            'game_id': game.get('game_id'),
            'game_date': _to_date_str(game.get('game_date')),
            'home_team': home_name,
            'away_team': away_name,
            'home_team_id': home_id,
            'away_team_id': away_id,
            'spread': float(out.get('spread', 0.0)),
            'q10': float(out.get('q10', 0.0)),
            'q90': float(out.get('q90', 0.0)),
            'uncertainty': float(out.get('uncertainty', 0.0)),
            'win_prob': float(out.get('win_prob', 0.5)),
            'win_prob_raw': float(out.get('win_prob', 0.5)),
            'win_prob_calibrated': float(out.get('win_prob_calibrated', out.get('win_prob', 0.5))),
            'confidence': str(out.get('confidence', 'LOW')),
            'new_feature': f"elo_bucket:{'home' if derived_features.get('elo_diff', 0.0) >= 0 else 'away'}|rest:{derived_features.get('rest_diff', 0.0):.1f}",
            'model_contributions': _to_native(out.get('model_contributions', {})),
        }
        pred.update(derived_features)

        feature_snapshot_native = _to_native(features_df.iloc[0].to_dict())
        feature_snapshot = feature_snapshot_native if isinstance(feature_snapshot_native, dict) else {}
        feature_snapshot.update({k: float(v) for k, v in derived_features.items()})

        pred_id = feedback.log_prediction(
            prediction=pred,
            features=feature_snapshot,
            game_date_utc=game.get('game_date'),
            new_feature=pred.get('new_feature'),
        )
        logged_prediction_ids.append(pred_id)
        predictions.append(pred)

print('Predictions generated:', len(predictions))
print('Predictions logged to SQLite with timezone fields:', len(logged_prediction_ids))

# 5) Increment batch counter if we actually logged predictions
if logged_prediction_ids:
    with SportsAnalyticsDB(DB_PATH) as db:
        state = db.get_retraining_state()
        prev_count = int(state.get('incremental_count') or 0)
        next_count = prev_count + 1
        db.update_retraining_state(incremental_count=next_count)
    print(f'Incremented prediction-batch counter: {prev_count} -> {next_count}')
else:
    print('No predictions logged; counter unchanged.')

pd.DataFrame([{
    'game_date': p.get('game_date'),
    'away_team': p.get('away_team'),
    'home_team': p.get('home_team'),
    'new_feature': p.get('new_feature'),
    'elo_diff': p.get('elo_diff'),
    'rest_diff': p.get('rest_diff'),
    'last5_win_pct_home': p.get('last5_win_pct_home'),
    'last5_win_pct_away': p.get('last5_win_pct_away'),
    'spread': p.get('spread'),
    'win_prob_raw': p.get('win_prob_raw'),
    'win_prob_calibrated': p.get('win_prob_calibrated'),
    'q10': p.get('q10'),
    'q90': p.get('q90'),
    'uncertainty': p.get('uncertainty'),
    'confidence': p.get('confidence'),
} for p in predictions])

Calibrator fit status: {'fitted': 1.0, 'limit': 200.0, 'min_samples': 10.0}
  Fetching 2023-24...
    -> 2460 records
  Fetching 2024-25...
    -> 2460 records
  Total: 4920 records, 2023-10-24 -> 2025-04-13
  Building matchup features...
    2445 training rows, 2023-10-26 00:00:00 -> 2025-04-13 00:00:00
    24 feature columns
Using model feature columns: 38
  Found 10 upcoming games
Predictions generated: 10
Predictions logged to SQLite with timezone fields: 10
Incremented prediction-batch counter: 4 -> 5


,game_date,away_team,home_team,new_feature,elo_diff,rest_diff,last5_win_pct_home,last5_win_pct_away,spread,win_prob_raw,win_prob_calibrated,q10,q90,uncertainty,confidence
0,2026-04-08,Oklahoma City Thunder,Los Angeles Lakers,elo_bucket:away|rest:0.0,-166.372437,0.0,0.6,0.8,1.24,0.5008,0.6104,-15.99,16.56,8.1361,MEDIUM
1,2026-04-08,Dallas Mavericks,Los Angeles Clippers,elo_bucket:home|rest:0.0,167.000491,0.0,1.0,0.2,6.10,0.6468,0.6104,-11.63,21.79,8.3549,MEDIUM
2,2026-04-08,Sacramento Kings,Golden State Warriors,elo_bucket:home|rest:0.0,93.525965,0.0,0.4,0.6,1.34,0.5161,0.6104,-13.34,18.97,8.0774,MEDIUM
3,2026-04-07,Chicago Bulls,Washington Wizards,elo_bucket:away|rest:0.0,-234.787296,0.0,0.2,0.8,-1.97,0.4026,0.6104,-20.50,18.61,9.7783,MEDIUM
4,2026-04-07,Minnesota Timberwolves,Indiana Pacers,elo_bucket:home|rest:0.0,3.604001,0.0,0.8,0.8,4.52,0.6065,0.6104,-14.49,24.92,9.8518,MEDIUM
5,2026-04-07,Milwaukee Bucks,Brooklyn Nets,elo_bucket:away|rest:0.0,-242.823596,0.0,0.2,1.0,-3.98,0.3726,0.3839,-19.00,12.30,7.8249,MEDIUM
6,2026-04-07,Miami Heat,Toronto Raptors,elo_bucket:away|rest:0.0,-53.750458,0.0,0.4,0.4,3.48,0.5899,0.6104,-16.58,16.53,8.2776,MEDIUM
7,2026-04-08,Charlotte Hornets,Boston Celtics,elo_bucket:home|rest:0.0,426.796164,0.0,0.8,0.0,7.55,0.6861,0.6104,-9.98,23.30,8.3195,MEDIUM
8,2026-04-08,Utah Jazz,New Orleans Pelicans,elo_bucket:home|rest:0.0,70.209971,0.0,0.0,0.2,1.56,0.5129,0.6104,-15.95,15.45,7.8506,MEDIUM
9,2026-04-08,Houston Rockets,Phoenix Suns,elo_bucket:away|rest:0.0,-132.047487,0.0,0.2,0.4,-2.10,0.4202,0.6104,-18.73,14.66,8.3463,MEDIUM


In [23]:
# PHASE 2 DIAGNOSTIC: Confirm games_df column structure
# This validates the vectorized feature function will work correctly

print("=== GAMES_DF STRUCTURE DIAGNOSIS ===")
print(f"Shape: {games_df.shape}")
print(f"\nColumns: {list(games_df.columns)}")
print(f"\nData types:\n{games_df[['TEAM_ID', 'GAME_DATE', 'GAME_ID', 'WL', 'PTS']].dtypes}")
print(f"\nFirst 3 rows:")
print(games_df[['TEAM_ID', 'GAME_DATE', 'GAME_ID', 'WL', 'PTS', 'MATCHUP']].head(3))
print(f"\nUnique teams: {games_df['TEAM_ID'].nunique()}")
print(f"Date range: {games_df['GAME_DATE'].min()} to {games_df['GAME_DATE'].max()}")
print("✓ Diagnostic complete - ready for vectorized feature function")


=== GAMES_DF STRUCTURE DIAGNOSIS ===
Shape: (4920, 28)

Columns: ['SEASON_ID', 'TEAM_ID', 'TEAM_ABBREVIATION', 'TEAM_NAME', 'GAME_ID', 'GAME_DATE', 'MATCHUP', 'WL', 'MIN', 'PTS', 'FGM', 'FGA', 'FG_PCT', 'FG3M', 'FG3A', 'FG3_PCT', 'FTM', 'FTA', 'FT_PCT', 'OREB', 'DREB', 'REB', 'AST', 'STL', 'BLK', 'TOV', 'PF', 'PLUS_MINUS']

Data types:
TEAM_ID               int64
GAME_DATE    datetime64[ns]
GAME_ID              object
WL                   object
PTS                   int64
dtype: object

First 3 rows:
      TEAM_ID  GAME_DATE     GAME_ID WL  PTS      MATCHUP
0  1610612737 2023-10-25  0022300063  L  110    ATL @ CHA
1  1610612737 2023-10-27  0022300079  L  120  ATL vs. NYK
2  1610612737 2023-10-29  0022300097  W  127    ATL @ MIL

Unique teams: 30
Date range: 2023-10-24 00:00:00 to 2025-04-13 00:00:00
✓ Diagnostic complete - ready for vectorized feature function


In [24]:
print("=" * 80)
print("PHASE 1: VERIFICATION — Game-Theory Feature Pipeline Health Check")
print("=" * 80)

# Ensure games_df is available
if 'games_df' not in locals() or games_df.empty:
    print("\n⚠️  WARNING: games_df is not available or empty.")
    print("   Skipping Phase 1 verification. Load data first.")
else:
    from src.evaluation.feedback_loop import derive_game_features
    
    # Test on a recent game matchup
    recent_sample = games_df.sort_values('GAME_DATE').tail(20)
    if len(recent_sample) > 0:
        home_team_id = int(recent_sample.iloc[-1].get('TEAM_ID', 1610612744))
        away_team_id = int(recent_sample[recent_sample['TEAM_ID'] != home_team_id].iloc[-1].get('TEAM_ID', 1610612747)) if len(recent_sample) > 1 else 1610612747
        test_date = pd.to_datetime(recent_sample.iloc[-1]['GAME_DATE'])
        
        test_game = {
            'game_date': test_date,
            'home_team_id': home_team_id,
            'away_team_id': away_team_id,
        }
        
        try:
            features = derive_game_features(test_game, games_df)
            
            print(f"\n✓ Test game context:")
            print(f"  Date: {test_date.date()}")
            print(f"  Home Team ID: {home_team_id}, Away Team ID: {away_team_id}")
            print(f"\n✓ Game-Theory Features Status:")
            
            game_theory_features = [
                'expected_payoff_matrix',
                'optimal_path_delta',
                'signal_consistency_score',
            ]
            
            features_active = 0
            for feat in game_theory_features:
                val = features.get(feat, 0)
                is_active = val != 0 and val != 0.5
                is_default = val == 0 or val == 0.5
                
                if is_active:
                    status = "✅ ACTIVE (non-default)"
                    features_active += 1
                else:
                    status = "⚠️  DEFAULT (may be proxy-based)"
                
                print(f"  {feat}: {val:.6f} [{status}]")
            
            print(f"\n{'✅ READY' if features_active >= 2 else '⚠️  PARTIAL'}: {features_active}/3 features active")
            print("  Note: If all are defaults, play-by-play enrichment will significantly boost signal.")
            
        except Exception as exc:
            print(f"\n❌ ERROR during feature computation: {exc}")
            print("  Check that games_df has required columns: GAME_ID, TEAM_ID, GAME_DATE, PTS, etc.")
    else:
        print("\n⚠️  Not enough data in games_df to test. Ensure data is loaded.")

print("\n" + "=" * 80 + "\n")

PHASE 1: VERIFICATION — Game-Theory Feature Pipeline Health Check

✓ Test game context:
  Date: 2025-04-13
  Home Team ID: 1610612766, Away Team ID: 1610612750

✓ Game-Theory Features Status:
  expected_payoff_matrix: 527.530987 [✅ ACTIVE (non-default)]
  optimal_path_delta: 0.566140 [✅ ACTIVE (non-default)]
  signal_consistency_score: 0.500000 [⚠️  DEFAULT (may be proxy-based)]

✅ READY: 2/3 features active
  Note: If all are defaults, play-by-play enrichment will significantly boost signal.




In [25]:
# PHASE 2 VALIDATION: Quick test of vectorized feature function
# Run this to verify the new implementation works before full retrain

import sys
import time

# Force reload of vectorized_features to pick up the fix
for key in list(sys.modules.keys()):
    if 'vectorized_features' in key:
        del sys.modules[key]

from src.evaluation.vectorized_features import vectorize_high_signal_features
from src.evaluation.feedback_loop import HIGH_SIGNAL_FEATURE_COLUMNS

print("\n=== PHASE 2 VECTORIZATION TEST ===")
print(f"Testing with {len(matchup_df)} training rows and {len(games_df)} game rows...")

# Time the vectorized computation
start_time = time.time()
features_vec = vectorize_high_signal_features(
    matchup_df=matchup_df.head(100),  # Test on first 100 rows
    games_df=games_df,
    verbose=True
)
elapsed = time.time() - start_time

print(f"\n✓ Vectorized function works!")
print(f"  Output shape: {features_vec.shape}")
print(f"  Required features: {len(HIGH_SIGNAL_FEATURE_COLUMNS)}")
print(f"  Features present: {features_vec.shape[1]}")
print(f"  Sample row:\n{features_vec.iloc[0]}")
print(f"\nEstimated full dataset timing: {elapsed * len(matchup_df) / 100 / 60:.1f} minutes")
print(f"Previous method: ~4 hours (240 minutes)")
print(f"Expected speedup: {240 * 60 / (elapsed * len(matchup_df) / 100):.0f}x")


=== PHASE 2 VECTORIZATION TEST ===
Testing with 2445 training rows and 4920 game rows...
[Vectorize] Processing 100 matchups with 4920 game rows
  Computing rolling 5-game stats...
  Computing rest days...
  Computing home/away strength...
  Computing schedule density...
  Computing pace...
  Computing ELO ratings...
  Merging home team features...
  Merging away team features...
  Computing matchup deltas...
  Building output...
✓ Vectorized feature computation complete in 13.5s
  Output shape: (100, 17)

✓ Vectorized function works!
  Output shape: (100, 17)
  Required features: 17
  Features present: 17
  Sample row:
elo_diff                    -20.000000
last5_win_pct_home            0.000000
last5_win_pct_away            1.000000
last5_point_diff_home       107.000000
last5_point_diff_away       108.000000
rest_days_home                2.000000
rest_days_away                2.000000
rest_diff                     0.000000
is_back_to_back_home          0.000000
is_back_to_back_away

In [26]:
import os
import sys

from IPython.display import display, HTML

# Force reload so this cell always picks up latest feedback_loop.py edits.
# If you changed package structure/imports (not just function internals), restart kernel.
for key in list(sys.modules.keys()):
    if key == 'src.evaluation.feedback_loop' or key.startswith('src.evaluation.feedback_loop'):
        del sys.modules[key]

from src.evaluation.feedback_loop import (
    fetch_and_update_results,
    evaluate_recent_predictions,
    get_recent_predictions,
    get_accuracy_summary,
    render_accuracy_summary_html,
    log_feature_metrics_to_mlflow,
)

def _print_detailed_diagnostics(summary: dict, evaluation: dict) -> None:
    """Print practical diagnostics for calibration, confidence quality, and drift hints."""
    overall_acc = float(summary.get('accuracy_pct', 0.0)) / 100.0

    by_conf = summary.get('by_confidence') or {}
    if by_conf:
        print('Confidence diagnostics:')
        conf_flags = []
        for level, row in by_conf.items():
            count = int(row.get('count', 0))
            acc = float(row.get('accuracy', 0.0))
            avg_pred = float(row.get('avg_predicted_probability', 0.0))
            gap = abs(avg_pred - acc)
            print(f"  {level}: n={count}, acc={acc * 100:.1f}%, avg_pred={avg_pred * 100:.1f}%, gap={gap * 100:.1f} pp")
            if count >= 8 and gap >= 0.10:
                conf_flags.append((level, gap, count))
        if conf_flags:
            print('Calibration warning by confidence level (>=8 samples and >=10pp gap):')
            for level, gap, count in conf_flags:
                print(f"  - {level}: gap={gap * 100:.1f} pp over {count} samples")

    calibration_rows = evaluation.get('calibration') or []
    calibration_candidates = []
    for row in calibration_rows:
        count = int(row.get('count', 0))
        if count < 5:
            continue
        avg_pred = float(row.get('avg_predicted_probability', 0.0))
        actual_rate = float(row.get('actual_home_win_rate', 0.0))
        gap = abs(avg_pred - actual_rate)
        calibration_candidates.append((gap, str(row.get('bucket')), count, avg_pred, actual_rate))

    if calibration_candidates:
        worst_gap, worst_bucket, worst_count, worst_pred, worst_actual = max(
            calibration_candidates,
            key=lambda x: x[0],
        )
        print(
            f"Calibration max gap (>=5 samples): bucket={worst_bucket}, gap={worst_gap * 100:.1f} pp "
            f"(pred={worst_pred * 100:.1f}%, actual={worst_actual * 100:.1f}%, n={worst_count})"
        )
        if worst_gap >= 0.12:
            print(
                'Hint: probabilities are miscalibrated in at least one band; consider probability calibration '
                '(Platt/isotonic) and confidence-threshold tuning.'
            )

    weak_features = []
    for feature_value, row in (summary.get('by_new_feature') or {}).items():
        count = int(row.get('count', 0))
        if count < 5:
            continue
        acc = float(row.get('accuracy', 0.0))
        rolling_latest = float(row.get('rolling_accuracy_latest', acc))
        mae_spread = float(row.get('mae_spread', 0.0))
        if acc <= overall_acc - 0.08 or rolling_latest <= acc - 0.08:
            weak_features.append((str(feature_value), count, acc, rolling_latest, mae_spread))

    if weak_features:
        weak_features = sorted(weak_features, key=lambda x: (x[2], x[3]))[:5]
        print('Potential weak/drifting feature segments (top 5):')
        for feature_value, count, acc, rolling_latest, mae_spread in weak_features:
            print(
                f"  - {feature_value}: n={count}, acc={acc * 100:.1f}%, "
                f"rolling={rolling_latest * 100:.1f}%, mae={mae_spread:.2f}"
            )
        print(
            'Hint: prioritize feature engineering and data checks for these segments; consider segment-specific '
            'weights or fallback logic for low-performing buckets.'
        )
    elif summary.get('by_new_feature'):
        print('No major feature-segment drift flags detected with current thresholds.')

RETRAIN_EVERY_N = int(globals().get('RETRAIN_EVERY_N', 7) or 7)

# Update unresolved predictions with final game outcomes (cache first, then nba_api fallback).
result_update = fetch_and_update_results(db_path=DB_PATH, days=14, verbose=True)
print('Result update:', result_update)

diag = result_update.get('diagnostics', {})
if diag:
    print('Result lookup inventory:', diag.get('inputs', {}))
    print('Resolution sources:', diag.get('resolved', {}))
    print('Unresolved reason buckets:', diag.get('unresolved_reasons', {}))

if result_update.get('pending_or_future', 0):
    print(
        'Note: pending/future games still have actual_winner = Pending until the final score is available:',
        result_update.get('pending_or_future', 0),
    )
if result_update.get('unresolved_past', 0):
    print('Warning: unresolved past games:', result_update.get('unresolved_past', 0))
    print('Examples:', result_update.get('unresolved_examples', [])[:5])

recent_feedback = get_recent_predictions(db_path=DB_PATH, n=20)
display(recent_feedback)

accuracy_summary = get_accuracy_summary(db_path=DB_PATH, n=100, retrain_every_n=RETRAIN_EVERY_N)
evaluation_metrics = evaluate_recent_predictions(db_path=DB_PATH, n=100)

print('Summary:', accuracy_summary.get('headline', accuracy_summary))

comparison = accuracy_summary.get('baseline_comparison', {}) or {}
print('=== Baseline Comparison ===')
print(comparison)

brier_raw = accuracy_summary.get('brier_raw')
brier_calibrated = accuracy_summary.get('brier_calibrated', accuracy_summary.get('brier_score'))
print(f'Brier raw: {float(brier_raw) if brier_raw is not None else float("nan"):.4f}')
print(f'Brier calibrated: {float(brier_calibrated) if brier_calibrated is not None else float("nan"):.4f}')

retrain_progress = accuracy_summary.get('retrain_progress', {}) or {}
progress_count = int(retrain_progress.get('progress_count', 0))
retrain_every_n = int(retrain_progress.get('retrain_every_n', RETRAIN_EVERY_N))
remaining_batches = int(retrain_progress.get('remaining_batches', retrain_every_n))
ready_to_retrain = bool(retrain_progress.get('ready_to_retrain', False))

if retrain_progress:
    print(f'Retrain progress: {progress_count}/{retrain_every_n} prediction batches since last retrain')
    print('Batches remaining until next auto full retrain:', remaining_batches)
    if ready_to_retrain:
        print('Retrain status: due now (next prediction batch should trigger full retrain).')

_print_detailed_diagnostics(accuracy_summary, evaluation_metrics)

display(pd.DataFrame(accuracy_summary.get('by_confidence', {})).T)
display(pd.DataFrame(accuracy_summary.get('by_new_feature', {})).T)
display(pd.DataFrame(accuracy_summary.get('by_elo_bucket', {})).T)
display(pd.DataFrame(accuracy_summary.get('by_rest_diff_bucket', {})).T)
display(pd.DataFrame(accuracy_summary.get('by_last5_form_bucket', {})).T)
display(pd.DataFrame(evaluation_metrics.get('calibration', [])))

# Pretty HTML block with color-coded feature impact and retrain progress.
display(HTML(render_accuracy_summary_html(accuracy_summary)))

# MLflow setup (env-driven):
#   MLFLOW_ENABLED=true
#   MLFLOW_TRACKING_URI=file:./mlruns
#   MLFLOW_EXPERIMENT_NAME=nba-prediction-feedback
ENABLE_MLFLOW = os.getenv('MLFLOW_ENABLED', 'false').strip().lower() in ('1', 'true', 'yes', 'on')
MLFLOW_TRACKING_URI = os.getenv('MLFLOW_TRACKING_URI', 'file:./mlruns')
MLFLOW_EXPERIMENT_NAME = os.getenv('MLFLOW_EXPERIMENT_NAME', 'nba-prediction-feedback')

print('Python executable:', sys.executable)
print('MLflow enabled (MLFLOW_ENABLED):', ENABLE_MLFLOW)
print('MLflow tracking URI:', MLFLOW_TRACKING_URI)
print('MLflow experiment:', MLFLOW_EXPERIMENT_NAME)

if ENABLE_MLFLOW:
    try:
        import mlflow

        mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
        mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)
        print('MLflow configuration applied.')
    except Exception as exc:
        print(f'MLflow configuration failed: {exc}')
        ENABLE_MLFLOW = False

mlflow_status = log_feature_metrics_to_mlflow(
    summary=accuracy_summary,
    run_name='nba-feedback-loop',
    enable=ENABLE_MLFLOW,
    tags={
        'stage': 'notebook',
        'component': 'feedback_loop',
        'tracking_uri': MLFLOW_TRACKING_URI,
        'experiment': MLFLOW_EXPERIMENT_NAME,
        'retrain_progress': f'{progress_count}/{retrain_every_n}',
    },
)
print('MLflow status:', mlflow_status)
if mlflow_status.get('status') == 'skipped':
    reason = mlflow_status.get('reason')
    if reason == 'enable=False':
        print('MLflow note: set MLFLOW_ENABLED=true to enable tracking for this cell run.')
    else:
        print(f"MLflow note: skipped due to configuration/import issue -> {reason}")
elif mlflow_status.get('status') == 'logged':
    print('MLflow logging successful.')

Checking unsettled predictions: 20
Result lookup seasons: ['2025-26']
Lookup inventory: {'cache_game_ids': 3690, 'api_game_ids': 1179, 'api_matchup_keys': 1179}
Result update: {'updated': 14, 'skipped': 6, 'pending_or_future': 6, 'unresolved_past': 0, 'resolved_from_cache': 0, 'resolved_from_api_game_id': 14, 'resolved_from_api_matchup': 0, 'unresolved_reason_counts': {'cache_game_id_not_found': 6}, 'unresolved_examples': [{'prediction_id': 90, 'game_id': '0022501155', 'local_date': '2026-04-07', 'matchup': 'Oklahoma City Thunder @ Los Angeles Lakers', 'reason': 'cache_game_id_not_found', 'pending_or_future': True}, {'prediction_id': 91, 'game_id': '0022501156', 'local_date': '2026-04-07', 'matchup': 'Dallas Mavericks @ Los Angeles Clippers', 'reason': 'cache_game_id_not_found', 'pending_or_future': True}, {'prediction_id': 99, 'game_id': '0022501157', 'local_date': '2026-04-07', 'matchup': 'Houston Rockets @ Phoenix Suns', 'reason': 'cache_game_id_not_found', 'pending_or_future': True

,game_date_pst,matchup,new_feature,elo_diff,rest_diff,last5_win_pct_home,last5_win_pct_away,last5_point_diff_home,last5_point_diff_away,predicted_winner,actual_winner,probability_raw,probability_calibrated,probability,correct
0,20:00,Houston Rockets @ Phoenix Suns,elo_bucket:away|rest:0.0,-132.047487,0.0,0.2,0.4,6.0,2.6,Houston Rockets,Pending,0.4202,0.6104,0.6104,NaN
1,17:00,Utah Jazz @ New Orleans Pelicans,elo_bucket:home|rest:0.0,70.209971,0.0,0.0,0.2,0.2,7.0,New Orleans Pelicans,New Orleans Pelicans,0.5129,0.6104,0.6104,1.0
2,17:00,Charlotte Hornets @ Boston Celtics,elo_bucket:home|rest:0.0,426.796164,0.0,0.8,0.0,-10.6,2.6,Boston Celtics,Boston Celtics,0.6861,0.6104,0.6104,1.0
3,16:30,Miami Heat @ Toronto Raptors,elo_bucket:away|rest:0.0,-53.750458,0.0,0.4,0.4,-3.8,5.8,Toronto Raptors,Toronto Raptors,0.5899,0.6104,0.6104,1.0
4,16:30,Milwaukee Bucks @ Brooklyn Nets,elo_bucket:away|rest:0.0,-242.823596,0.0,0.2,1.0,-2.4,-0.6,Milwaukee Bucks,Brooklyn Nets,0.3726,0.3839,0.3839,0.0
5,16:00,Minnesota Timberwolves @ Indiana Pacers,elo_bucket:home|rest:0.0,3.604001,0.0,0.8,0.8,1.8,2.2,Indiana Pacers,Minnesota Timberwolves,0.6065,0.6104,0.6104,0.0
6,16:00,Chicago Bulls @ Washington Wizards,elo_bucket:away|rest:0.0,-234.787296,0.0,0.2,0.8,1.8,1.8,Chicago Bulls,Chicago Bulls,0.4026,0.6104,0.6104,1.0
7,19:00,Sacramento Kings @ Golden State Warriors,elo_bucket:home|rest:0.0,93.525965,0.0,0.4,0.6,1.4,-1.6,Golden State Warriors,Golden State Warriors,0.5161,0.6104,0.6104,1.0
8,19:30,Dallas Mavericks @ Los Angeles Clippers,elo_bucket:home|rest:0.0,167.000491,0.0,1.0,0.2,-0.8,5.6,Los Angeles Clippers,Pending,0.6468,0.6104,0.6104,NaN
9,19:30,Oklahoma City Thunder @ Los Angeles Lakers,elo_bucket:away|rest:0.0,-166.372437,0.0,0.6,0.8,-4.2,-1.0,Los Angeles Lakers,Pending,0.5008,0.6104,0.6104,NaN


Summary: Accuracy: 59.0% over last 100 settled predictions
=== Baseline Comparison ===
{'accuracy_delta': 0.06219999999999992, 'brier_delta': 0.02972294090000005, 'mae_delta': 0.2201999999999984}
Brier raw: 0.2561
Brier calibrated: 0.2609
Retrain progress: 5/7 prediction batches since last retrain
Batches remaining until next auto full retrain: 2
Confidence diagnostics:
  MEDIUM: n=100, acc=59.0%, avg_pred=61.2%, gap=2.2 pp
Calibration max gap (>=5 samples): bucket=(0.4, 0.5], gap=44.3 pp (pred=44.3%, actual=0.0%, n=6)
Hint: probabilities are miscalibrated in at least one band; consider probability calibration (Platt/isotonic) and confidence-threshold tuning.
Potential weak/drifting feature segments (top 5):
  - spread_bucket:underdog|conf:MEDIUM: n=11, acc=36.4%, rolling=36.4%, mae=16.56
  - spread_bucket:heavy_favorite|conf:MEDIUM: n=7, acc=42.9%, rolling=42.9%, mae=15.92
  - spread_bucket:moderate_favorite|conf:MEDIUM: n=9, acc=44.4%, rolling=44.4%, mae=14.21
Hint: prioritize featur

,count,accuracy,avg_predicted_probability
MEDIUM,100.0,0.59,0.611801


,count,accuracy,brier_raw,brier_calibrated,brier_score,mae_spread,rolling_accuracy_latest,calibration
elo_bucket:away|rest:0.0,23,0.565217,0.234662,0.281886,0.281886,12.391739,0.55,"[{'bucket': '(0.2, 0.4]', 'count': 3, 'avg_pre..."
elo_bucket:home|rest:0.0,35,0.657143,0.24553,0.227061,0.227061,14.568571,0.65,"[{'bucket': '(0.4, 0.6]', 'count': 15, 'avg_pr..."
spread_bucket:coin_flip|conf:MEDIUM,15,0.8,0.22164,0.214173,0.214173,6.771333,0.8,"[{'bucket': '(0.4, 0.6]', 'count': 15, 'avg_pr..."
spread_bucket:heavy_favorite|conf:MEDIUM,7,0.428571,0.315253,0.329235,0.329235,15.918571,0.428571,"[{'bucket': '(0.6, 0.8]', 'count': 7, 'avg_pre..."
spread_bucket:moderate_favorite|conf:MEDIUM,9,0.444444,0.261408,0.267225,0.267225,14.207778,0.444444,"[{'bucket': '(0.4, 0.6]', 'count': 5, 'avg_pre..."
spread_bucket:underdog|conf:MEDIUM,11,0.363636,0.339539,0.339539,0.339539,16.563636,0.363636,"[{'bucket': '(0.2, 0.4]', 'count': 10, 'avg_pr..."


...

,count,accuracy,brier_calibrated
away_slight_adv,58.0,0.62069,0.248802


,count,accuracy,brier_calibrated
away_hot,15.0,0.600000,0.269991
away_edge,3.0,0.666667,0.240772
neutral_form,55.0,0.581818,0.262923
home_edge,9.0,0.444444,0.315067
home_hot,18.0,0.666667,0.223287


,bucket,count,avg_predicted_probability,actual_home_win_rate
0,"(0.2, 0.3]",4,0.281050,0.500000
1,"(0.3, 0.4]",9,0.354600,0.777778
2,"(0.4, 0.5]",6,0.442617,0.000000
3,"(0.5, 0.6]",14,0.566336,0.500000
4,"(0.6, 0.7]",64,0.631083,0.593750
5,"(0.7, 0.8]",3,0.719200,0.333333


new_feature,Count,Accuracy,Brier,MAE
elo_bucket:away|rest:0.0,23,56.5%,0.2819,12.39
elo_bucket:home|rest:0.0,35,65.7%,0.2271,14.57
spread_bucket:coin_flip|conf:MEDIUM,15,80.0%,0.2142,6.77
spread_bucket:heavy_favorite|conf:MEDIUM,7,42.9%,0.3292,15.92
spread_bucket:moderate_favorite|conf:MEDIUM,9,44.4%,0.2672,14.21
spread_bucket:underdog|conf:MEDIUM,11,36.4%,0.3395,16.56


Python executable: c:\Users\Windows User\AppData\Local\Programs\Python\Python313\python.exe
MLflow enabled (MLFLOW_ENABLED): False
MLflow tracking URI: file:./mlruns
MLflow experiment: nba-prediction-feedback
MLflow status: {'status': 'skipped', 'reason': 'enable=False'}
MLflow note: set MLFLOW_ENABLED=true to enable tracking for this cell run.


In [27]:
from IPython.display import display, HTML

from reports.html_report import generate as generate_html

ts = datetime.now().strftime('%Y-%m-%d %H:%M')
title = f'NBA Predictions ({ts})'

display(HTML(generate_html(predictions, title=title)))

#,Date,Matchup,Favored,Spread,Win Prob,Interval,Uncertainty,Confidence
1,2026-04-08,Oklahoma City Thunder @ Los Angeles Lakers,Los Angeles Lakers,1.24,50.1%,"[-15.99, 16.56]",8.136,MEDIUM
2,2026-04-08,Dallas Mavericks @ Los Angeles Clippers,Los Angeles Clippers,6.10,64.7%,"[-11.63, 21.79]",8.355,MEDIUM
3,2026-04-08,Sacramento Kings @ Golden State Warriors,Golden State Warriors,1.34,51.6%,"[-13.34, 18.97]",8.077,MEDIUM
4,2026-04-07,Chicago Bulls @ Washington Wizards,Chicago Bulls,-1.97,40.3%,"[-20.50, 18.61]",9.778,MEDIUM
5,2026-04-07,Minnesota Timberwolves @ Indiana Pacers,Indiana Pacers,4.52,60.7%,"[-14.49, 24.92]",9.852,MEDIUM
6,2026-04-07,Milwaukee Bucks @ Brooklyn Nets,Milwaukee Bucks,-3.98,37.3%,"[-19.00, 12.30]",7.825,MEDIUM
7,2026-04-07,Miami Heat @ Toronto Raptors,Toronto Raptors,3.48,59.0%,"[-16.58, 16.53]",8.278,MEDIUM
8,2026-04-08,Charlotte Hornets @ Boston Celtics,Boston Celtics,7.55,68.6%,"[-9.98, 23.30]",8.319,MEDIUM
9,2026-04-08,Utah Jazz @ New Orleans Pelicans,New Orleans Pelicans,1.56,51.3%,"[-15.95, 15.45]",7.851,MEDIUM
10,2026-04-08,Houston Rockets @ Phoenix Suns,Houston Rockets,-2.10,42.0%,"[-18.73, 14.66]",8.346,MEDIUM


In [28]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

print("=" * 80)
print("STRATEGIC TENSION VISUALIZATION")
print("Subgame Perfect Equilibrium Analysis: Optimal vs. Observed Play")
print("=" * 80)

if 'predictions' not in locals() or not predictions:
    print("\n⚠️  No predictions available. Run prediction cells first.")
else:
    # Extract game-theory metrics from prediction logs
    pred_data = []
    for idx, pred in enumerate(predictions):
        game_id = pred.get('game_id', f'Game_{idx}')
        date_str = pred.get('when_prediction_made', '')
        features = pred.get('features', {})
        pred_prob = float(pred.get('pred_home_win_prob', 0.5))
        
        optimal_delta = features.get('optimal_path_delta', 0.0)
        signal_score = features.get('signal_consistency_score', 0.5)
        expected_payoff = features.get('expected_payoff_matrix', 0.0)
        
        pred_data.append({
            'game_id': game_id,
            'date': date_str,
            'optimal_delta': optimal_delta,
            'signal_score': signal_score,
            'expected_payoff': expected_payoff,
            'pred_win_prob': pred_prob,
            'confidence': 1.0 - abs(pred_prob - 0.5) * 2,  # Confidence near 0 or 1
        })
    
    if pred_data:
        df_viz = pd.DataFrame(pred_data)
        
        # Create subplots: optimal_delta, signal_score, and confidence
        fig = make_subplots(
            rows=3, cols=1,
            subplot_titles=(
                "Optimal Path Delta (0=equilibrium, 1=maximal deviation)",
                "Signal Consistency Score (1=aligned, 0.5=weak, 0=conflicted)",
                "Model Confidence in Prediction"
            ),
            specs=[[{}], [{}], [{}]],
            shared_xaxes=True,
            vertical_spacing=0.12,
        )
        
        # Row 1: Optimal Delta (Green = aligned to equilibrium, Red = deviation)
        colors_delta = ['green' if x < 0.3 else 'orange' if x < 0.6 else 'red' for x in df_viz['optimal_delta']]
        fig.add_trace(
            go.Bar(
                x=df_viz.index,
                y=df_viz['optimal_delta'],
                marker_color=colors_delta,
                name='Optimal Path Delta',
                hovertemplate='<b>Game %{x}</b><br>Optimal Delta: %{y:.3f}<extra></extra>',
            ),
            row=1, col=1,
        )
        fig.add_hline(y=0.3, line_dash="dash", line_color="green", annotation_text="Low Deviation Zone", row=1, col=1)
        fig.add_hline(y=0.6, line_dash="dash", line_color="red", annotation_text="High Deviation Zone", row=1, col=1)
        
        # Row 2: Signal Consistency Score
        colors_signal = ['green' if x > 0.7 else 'orange' if x > 0.5 else 'red' for x in df_viz['signal_score']]
        fig.add_trace(
            go.Scatter(
                x=df_viz.index,
                y=df_viz['signal_score'],
                mode='lines+markers',
                marker=dict(size=8, color=colors_signal),
                name='Signal Consistency',
                hovertemplate='<b>Game %{x}</b><br>Signal Score: %{y:.3f}<extra></extra>',
            ),
            row=2, col=1,
        )
        fig.add_hline(y=0.5, line_dash="dash", line_color="gray", annotation_text="Baseline", row=2, col=1)
        
        # Row 3: Model Confidence
        fig.add_trace(
            go.Scatter(
                x=df_viz.index,
                y=df_viz['confidence'],
                fill='tozeroy',
                mode='lines',
                name='Prediction Confidence',
                line_color='blue',
                hovertemplate='<b>Game %{x}</b><br>Confidence: %{y:.1%}<extra></extra>',
            ),
            row=3, col=1,
        )
        
        # Update layout
        fig.update_yaxes(title_text="Delta (0-1)", row=1, col=1)
        fig.update_yaxes(title_text="Score (0-1)", row=2, col=1)
        fig.update_yaxes(title_text="Confidence (0-1)", row=3, col=1)
        fig.update_xaxes(title_text="Game Index", row=3, col=1)
        
        fig.update_layout(
            title_text="<b>Strategic Tension: Coaching Logic vs. Game Theory Logic</b>",
            title_font_size=16,
            height=900,
            template="plotly_white",
            hovermode="x unified",
            showlegend=True,
        )
        
        fig.show()
        
        # Print summary statistics
        print(f"\n📊 Strategic Tension Summary ({len(df_viz)} predictions):")
        print(f"  Avg Optimal Delta: {df_viz['optimal_delta'].mean():.3f} (0=pure equilibrium play)")
        print(f"  Avg Signal Score:  {df_viz['signal_score'].mean():.3f} (1.0=strongest signal)")
        print(f"  Avg Confidence:    {df_viz['confidence'].mean():.1%}")
        print(f"\n  🔍 Interpretation:")
        print(f"    • Low Delta + High Signal = Teams playing optimally per game theory")
        print(f"    • High Delta + Low Signal = Coaching deviation from equilibrium (opportunity?)")
        print(f"    • High Confidence = Clear predictor signal (low uncertainty)")
    else:
        print("\n⚠️  No prediction data extracted. Check prediction structure.")

print("\n" + "=" * 80 + "\n")

STRATEGIC TENSION VISUALIZATION
Subgame Perfect Equilibrium Analysis: Optimal vs. Observed Play



📊 Strategic Tension Summary (10 predictions):
  Avg Optimal Delta: 0.000 (0=pure equilibrium play)
  Avg Signal Score:  0.500 (1.0=strongest signal)
  Avg Confidence:    100.0%

  🔍 Interpretation:
    • Low Delta + High Signal = Teams playing optimally per game theory
    • High Delta + Low Signal = Coaching deviation from equilibrium (opportunity?)
    • High Confidence = Clear predictor signal (low uncertainty)


